# Matplotlib — plotting and animation

**What this library does in one sentence:** Matplotlib turns arrays of numbers into 2-D pictures — static plots, multi-panel figures, and frame-by-frame animations.

For the cart-pole project we use it for: time-series plots of the trajectory, phase portraits of `(θ, θ̇)`, and notebook-embedded animations of the pendulum swinging.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## 1. The figure / axes mental model

Every matplotlib plot has three things, nested:

```
Figure   ←  the whole picture / window
  Axes   ←  one coordinate system (one "plot"). A figure can have many.
    Artist  ←  everything drawn inside: lines, markers, text, legend
```

A `Figure` is the outer canvas. An `Axes` is a single plot area inside it — the thing with x and y axes. `Artists` are the things you draw on the axes.

**`plt.plot(...)` is a shortcut** that creates a figure + axes if there isn't one yet, then draws on it. For anything beyond a quick look you should grab the axes explicitly via `fig, ax = plt.subplots()`.

## 2. Basic plotting

### `plt.plot(x, y, ...)` — line plot

| Parameter   | Meaning |
|-------------|---------|
| `x`, `y`    | 1-D arrays of equal length. |
| `color`     | Colour string: `'red'`, `'C0'`, `'#5DCAA5'`, or `(r, g, b)` tuple. |
| `linewidth` | Line thickness (default ~1.5). |
| `linestyle` | `'-'` solid, `'--'` dashed, `':'` dotted, `'-.'` dash-dot. |
| `label`     | String used by `legend()`. |
| `alpha`     | Transparency, 0 (invisible) to 1 (opaque). |

In [ ]:
# minimal
x = np.linspace(0, 2*np.pi, 200)
plt.plot(x, np.sin(x))
plt.show()

# pendulum — plot a fake theta(t) trajectory with multiple kwargs
t = np.linspace(0, 5, 500)
theta = 0.1 * np.exp(-0.2*t) * np.cos(3*t)

plt.plot(t, np.degrees(theta),
         color='#AFA9EC', linewidth=2,
         linestyle='-', label='pole angle', alpha=0.9)
plt.xlabel('t (s)'); plt.ylabel('theta (deg)')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

### `plt.scatter(x, y)` — point cloud

Same as `plot` but no line — just markers. Use it for sampled data, Poincaré sections, or initial-condition grids.

In [ ]:
# scatter a grid of initial conditions for a phase portrait
theta0 = np.radians(np.linspace(-30, 30, 7))
thetad0 = np.linspace(-2, 2, 5)
TH, TD = np.meshgrid(theta0, thetad0)
plt.scatter(np.degrees(TH), TD, color='#5DCAA5', s=30)
plt.xlabel('theta (deg)'); plt.ylabel('theta_dot (rad/s)')
plt.title('initial conditions to simulate'); plt.grid(alpha=0.3)
plt.show()

### `plt.axhline(y)` and `plt.axvline(x)` — guide lines

Draw a horizontal or vertical line across the whole axes. Useful for marking equilibria, thresholds, or `t=0`.

In [ ]:
plt.plot(t, np.degrees(theta))
plt.axhline(0,    color='white', linestyle='--', alpha=0.5)   # equilibrium
plt.axvline(2.5,  color='red',   linestyle=':',  alpha=0.7)   # mark t=2.5
plt.xlabel('t'); plt.ylabel('theta (deg)'); plt.grid(alpha=0.3)
plt.show()

## 3. Figure layout

### `plt.subplots(nrows, ncols, figsize)`

Creates a figure with a grid of axes. Returns `(fig, axes)`.

| Parameter | Meaning |
|-----------|---------|
| `nrows`   | Number of axes rows. |
| `ncols`   | Number of axes columns. |
| `figsize` | `(width, height)` in inches. |
| `sharex` / `sharey` | If `True`, link axes — pan/zoom together, only outermost labels show. |

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

x_traj  = 0.3 * np.sin(2*t)
th_traj = np.degrees(theta)

axes[0].plot(t, x_traj,  color='#5DCAA5')
axes[0].set_ylabel('cart x (m)')
axes[0].set_title('cart-pole trajectory')
axes[0].grid(alpha=0.3)

axes[1].plot(t, th_traj, color='#AFA9EC')
axes[1].set_ylabel('theta (deg)')
axes[1].set_xlabel('t (s)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Per-axes setters

When you have explicit `ax` objects, you call methods on them (note the `set_` prefix) instead of the global `plt.xlabel`.

| Method | Purpose |
|--------|---------|
| `ax.set_xlabel(s)`, `ax.set_ylabel(s)` | Axis labels. |
| `ax.set_title(s)`              | Title above the axes. |
| `ax.legend()`                  | Show the legend (uses `label=` from `plot`). |
| `ax.set_xlim(lo, hi)`, `ax.set_ylim(lo, hi)` | Manual axis ranges. |
| `ax.grid(True, alpha=0.3)`     | Toggle gridlines, with optional transparency. |
| `plt.tight_layout()`           | Auto-adjust spacing so labels don't overlap. |

## 4. Styling

### `plt.style.use('dark_background')`

Switches the global colour theme. Dark backgrounds look better for the pendulum visualisations (they match the pygame window) and project better on a screen.

In [ ]:
plt.style.use('dark_background')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, th_traj, color='#F0997B', linewidth=2)
ax.set_xlabel('t (s)'); ax.set_ylabel('theta (deg)')
ax.grid(alpha=0.2)
plt.show()

### Colour formats

Anywhere a `color=` is accepted, all three of these work:

```python
color='red'                # named
color='#5DCAA5'            # hex string  → mint green (this project's cart colour)
color=(0.36, 0.79, 0.65)   # RGB float tuple, 0–1
```

**Project palette** (matches the pygame window):

| Use         | Hex        |
|-------------|------------|
| cart        | `#5DCAA5`  |
| pole        | `#AFA9EC`  |
| bob / trail | `#F09A7B`  |
| phase line  | `#534AB7`  |
| force arrow | `#F0C850`  |

## 5. Animation with `FuncAnimation`

A matplotlib animation works by **drawing once, then modifying** the artists each frame. You don't redraw the whole figure — that would be slow. You hand back updated data to the same `Line2D` / `Circle` / `Text` objects.

### The artist pattern

```python
fig, ax = plt.subplots()
(line,) = ax.plot([], [], color='C0')   # create once, empty data

def init():
    ax.set_xlim(0, 10); ax.set_ylim(-2, 2)
    return (line,)                       # return iterable of artists to redraw

def update(i):
    line.set_data(t[:i], y[:i])          # mutate the existing artist
    return (line,)

anim = FuncAnimation(fig, update, frames=len(t), init_func=init, interval=20, blit=True)
```

### `FuncAnimation(fig, update, frames, init_func, interval, blit)` — full signature

| Parameter   | Meaning |
|-------------|---------|
| `fig`       | The Figure to animate. |
| `update`    | A function `update(i)` called per frame; `i` is the frame index (or whatever you yield from `frames`). Must return an iterable of artists to redraw (when `blit=True`). |
| `frames`    | An integer (`N` → call `update(0)` ... `update(N-1)`), or an iterable to step through. |
| `init_func` | Optional function called once, before the first frame. Set axis limits and initial artist state here. |
| `interval`  | Delay between frames in milliseconds. `interval=20` → ~50 fps. |
| `blit`      | If `True`, only redraw the artists returned by `update` (much faster). Requires `init_func`. |

### What is `blit`?

With `blit=False`, matplotlib redraws the entire figure every frame — slow.

With `blit=True`, matplotlib redraws *only* the artists you return from `update`. The static background (axes, grid, title) is painted once and reused. This is typically 5–10× faster — important for smooth playback.

The price: you must return every artist you modified, every frame, from `update` (and from `init_func`).

In [ ]:
# Full animation example — a swinging pendulum bob
L_pend = 0.8
t = np.linspace(0, 4, 200)
theta = 0.5 * np.cos(2*t)
x_bob =  L_pend * np.sin(theta)
y_bob = -L_pend * np.cos(theta)

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(-1, 1); ax.set_ylim(-1, 0.3); ax.set_aspect('equal')
ax.grid(alpha=0.2)

(rod,)   = ax.plot([], [], color='#AFA9EC', linewidth=3)
(bob,)   = ax.plot([], [], 'o', color='#F09A7B', markersize=14)
time_tx  = ax.text(0.02, 0.95, '', transform=ax.transAxes, color='white')

def init():
    rod.set_data([], [])
    bob.set_data([], [])
    time_tx.set_text('')
    return rod, bob, time_tx

def update(i):
    rod.set_data([0, x_bob[i]], [0, y_bob[i]])
    bob.set_data([x_bob[i]],    [y_bob[i]])
    time_tx.set_text(f't = {t[i]:.2f} s')
    return rod, bob, time_tx

anim = FuncAnimation(fig, update, frames=len(t),
                     init_func=init, interval=20, blit=True)
plt.close(fig)         # suppress duplicate static figure in the notebook

### Displaying in a notebook: `HTML(anim.to_jshtml())`

Jupyter doesn't show `FuncAnimation` objects directly. Convert to embedded HTML/JS:

```python
HTML(anim.to_jshtml())
```

The whole animation is encoded as base64 frames inside the notebook. Convenient — but **the notebook file grows huge fast** (every frame becomes ~kilobytes of base64). For anything over ~200 frames or a 5-minute animation, save to disk with `anim.save()` instead and link the file.

In [ ]:
# uncomment to display inline (warning: large notebook file)
# HTML(anim.to_jshtml())

### Saving with `anim.save()`

```python
anim.save('pendulum.gif', writer='pillow', fps=50)        # GIF — works everywhere, no ffmpeg
anim.save('pendulum.mp4', writer='ffmpeg', fps=50, dpi=150)  # MP4 — needs ffmpeg installed
```

- **`pillow`** writes GIFs. Slow, large files, but no system dependencies.
- **`ffmpeg`** writes MP4 / WebM / etc. Smaller, faster — but you need ffmpeg on your PATH.

For the cart-pole demo videos we use `ffmpeg`.

## 6. Phase portrait — specific to this project

A phase portrait plots `θ̇` vs `θ` — it strips out time and shows the *shape* of the motion. For a passive pendulum you get nested loops; for a damped one, spirals into the origin; for an inverted pendulum that falls over, lines escaping to infinity.

In [ ]:
# fake decaying trajectory in (theta, theta_dot)
t = np.linspace(0, 10, 1000)
theta  =  0.5 * np.exp(-0.15*t) * np.cos(2*t)
thetad = -0.5 * np.exp(-0.15*t) * (0.15*np.cos(2*t) + 2*np.sin(2*t))

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(np.degrees(theta), np.degrees(thetad),
        color='#534AB7', linewidth=1, alpha=0.8)
ax.scatter([np.degrees(theta[0])],  [np.degrees(thetad[0])],
           color='#5DCAA5', s=50, label='start')
ax.scatter([np.degrees(theta[-1])], [np.degrees(thetad[-1])],
           color='#F09A7B', s=50, label='end')
ax.axhline(0, color='white', linestyle='--', alpha=0.3)
ax.axvline(0, color='white', linestyle='--', alpha=0.3)
ax.set_xlabel('theta (deg)'); ax.set_ylabel('theta_dot (deg/s)')
ax.set_title('phase portrait — damped pendulum')
ax.legend(); ax.grid(alpha=0.2)
plt.show()

**For us:** phase portraits will show, side-by-side with the time plots, whether the LQR controller drives the state spiral neatly into the origin (good) or kicks it outward (bad).

## Functions used in this project — quick reference

| Function | Signature | What it does |
|----------|-----------|--------------|
| `plt.plot`         | `plt.plot(x, y, color=, linewidth=, linestyle=, label=, alpha=)` | Line plot. |
| `plt.scatter`      | `plt.scatter(x, y, s=, color=)`             | Marker plot, no line. |
| `plt.axhline`      | `plt.axhline(y, color=, linestyle=)`        | Horizontal guide line. |
| `plt.axvline`      | `plt.axvline(x, color=, linestyle=)`        | Vertical guide line. |
| `plt.subplots`     | `fig, ax = plt.subplots(nrows, ncols, figsize=, sharex=)` | Figure + axes grid. |
| `plt.tight_layout` | `plt.tight_layout()`                        | Auto-spacing. |
| `plt.style.use`    | `plt.style.use('dark_background')`           | Switch global theme. |
| `ax.set_xlabel`    | `ax.set_xlabel('t (s)')`                     | X-axis label. |
| `ax.set_ylabel`    | `ax.set_ylabel('theta')`                     | Y-axis label. |
| `ax.set_title`     | `ax.set_title('cart-pole')`                  | Axes title. |
| `ax.set_xlim`      | `ax.set_xlim(lo, hi)`                        | Manual x range. |
| `ax.set_ylim`      | `ax.set_ylim(lo, hi)`                        | Manual y range. |
| `ax.legend`        | `ax.legend()`                                | Show legend. |
| `ax.grid`          | `ax.grid(True, alpha=0.3)`                   | Gridlines. |
| `FuncAnimation`    | `FuncAnimation(fig, update, frames, init_func, interval, blit)` | Frame-by-frame animation. |
| `anim.save`        | `anim.save('out.mp4', writer='ffmpeg', fps=50)` | Write to disk. |
| `HTML(anim.to_jshtml())` | inline JS player                       | Display animation in a notebook. |